# Day 11 · Exercise 1: Spot the Similarity

**What you'll build:** `rank_by_similarity_score(pairs: list[tuple[str, str]]) -> list[float]` — a function that takes a list of sentence pairs and returns their cosine similarity scores using local embeddings.

**Why it matters:** Measuring semantic similarity is the atomic operation that powers semantic search, duplicate detection, and recommendation — once you can score pairs, you can rank anything by meaning.

## Your Implementation

In [ ]:
import math
import ollama

MODEL = "nomic-embed-text"


def rank_by_similarity_score(pairs: list[tuple[str, str]]) -> list[float]:
    """Return the cosine similarity score for each pair of strings.

    For each (text_a, text_b) pair, embed both strings using the local
    nomic-embed-text model and compute their cosine similarity.  The
    returned list preserves the same order as the input list.

    Args:
        pairs: A list of 2-tuples where each tuple holds two strings to
               compare.  Example: [("cat", "dog"), ("cat", "car")].

    Returns:
        A list of floats in [-1, 1], one per input pair.  Values close
        to 1.0 indicate high semantic similarity; values close to 0.0
        indicate unrelated meaning.

    Example:
        >>> scores = rank_by_similarity_score([
        ...     ("I love machine learning.", "Deep learning is fascinating."),
        ...     ("I love machine learning.", "The quarterly report is due Friday."),
        ... ])
        >>> scores[0]  # high — both about ML, e.g. 0.85
        >>> scores[1]  # low  — unrelated topics, e.g. 0.12
    """
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────

## Check Your Work

Run the cell below — it runs 4 automated checks and shows ✅ / ❌ for each.

In [ ]:
_PASS, _FAIL = '✅', '❌'


def _run_checks():
    score, total = 0, 4

    # Check 1: callable and returns a list
    try:
        assert callable(rank_by_similarity_score), 'rank_by_similarity_score is not defined'
        result = rank_by_similarity_score([("hello", "hello")])
        assert isinstance(result, list), f'expected list, got {type(result).__name__}'
        assert len(result) == 1, f'expected 1 element for 1 pair, got {len(result)}'
        print(f'{_PASS} Check 1/{total}: function exists, callable, returns a list')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 1/{total}: {e}')
        return

    # Check 2: identical strings score >= 0.99
    try:
        identical_score = rank_by_similarity_score(
            [("Embeddings are numeric vectors that encode meaning.",
              "Embeddings are numeric vectors that encode meaning.")]
        )[0]
        assert isinstance(identical_score, float), (
            f'score should be float, got {type(identical_score).__name__}'
        )
        assert identical_score >= 0.99, (
            f'identical strings should score >= 0.99, got {identical_score:.4f}'
        )
        print(f'{_PASS} Check 2/{total}: identical strings score >= 0.99 '
              f'(got {identical_score:.4f})')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 2/{total}: {e}')

    # Check 3: semantically similar pair scores >= 0.65
    try:
        similar_score = rank_by_similarity_score(
            [("The dog chased the ball across the park.",
              "A puppy ran after a toy in the garden.")]
        )[0]
        assert similar_score >= 0.65, (
            f'similar pair should score >= 0.65, got {similar_score:.4f}'
        )
        print(f'{_PASS} Check 3/{total}: similar-meaning pair scores >= 0.65 '
              f'(got {similar_score:.4f})')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 3/{total}: {e}')

    # Check 4: unrelated pair scores <= 0.4
    try:
        unrelated_score = rank_by_similarity_score(
            [("I love hiking in the mountains.",
              "The quarterly earnings report is due on Friday.")]
        )[0]
        assert unrelated_score <= 0.4, (
            f'unrelated pair should score <= 0.4, got {unrelated_score:.4f}'
        )
        print(f'{_PASS} Check 4/{total}: unrelated pair scores <= 0.4 '
              f'(got {unrelated_score:.4f})')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 4/{total}: {e}')

    print()
    if score == total:
        print('🎉 Exercise complete!')
        print(f'  {total}/{total} passed.')
    else:
        print(f'  {score}/{total} passed. Keep going!')


_run_checks()

## Bonus Challenge

Right now each call to `rank_by_similarity_score` embeds every string separately — even if the same string appears in multiple pairs.

Refactor your implementation to **deduplicate embeddings**: build a cache dict `{text: vector}` before computing similarities, so each unique string is embedded exactly once.

This foreshadows Day 14's batch-embedding pattern, where you'll embed hundreds of document chunks efficiently before building a search index.

## Solution

<details>
<summary>Click to reveal — try on your own first</summary>

```python
import math
import ollama

MODEL = "nomic-embed-text"


def _embed(text: str) -> list[float]:
    """Return the 768-dim embedding vector for text."""
    response = ollama.embeddings(model=MODEL, prompt=text)
    return response["embedding"]


def _cosine_similarity(a: list[float], b: list[float]) -> float:
    """Return cosine similarity in [-1, 1] between vectors a and b."""
    dot = sum(x * y for x, y in zip(a, b))
    mag_a = math.sqrt(sum(x * x for x in a))
    mag_b = math.sqrt(sum(x * x for x in b))
    return dot / (mag_a * mag_b)


def rank_by_similarity_score(pairs: list[tuple[str, str]]) -> list[float]:
    """Return cosine similarity for each (text_a, text_b) pair."""
    scores = []
    for text_a, text_b in pairs:
        vec_a = _embed(text_a)
        vec_b = _embed(text_b)
        scores.append(_cosine_similarity(vec_a, vec_b))
    return scores
```

**Why this works:** each string is passed to `ollama.embeddings`, which returns a 768-dimensional vector whose position in space encodes the string's meaning. Cosine similarity then measures the angle between the two vectors — texts that share topics and vocabulary point in nearly the same direction (score near 1), while unrelated texts point in very different directions (score near 0). The loop preserves input order so callers can zip scores back against their original pairs.
</details>